# O2 A Plot Accessor Demo

This notebook is a catalog of the supported plotting surface: `.plot` accessors on zdisamar domain objects. Each plot is displayed in its own cell.


## Setup

The setup prepares one O2 A case, runs the native diagnostics, and stores one Altair chart per public accessor method.


In [1]:
import zdisamar as zd

from validation.common import o2a_retrieval_baseline as oe_baseline
from validation.common.o2a_measurement_noise import components_from_spectrum

O2A_MARKERS_NM = (
    oe_baseline.WAVELENGTH_START_NM,
    760.76,
    oe_baseline.WAVELENGTH_END_NM,
)


def spectral_grid(case) -> list[float]:
    start = float(case.spectral_grid.start_nm)
    end = float(case.spectral_grid.end_nm)
    count = int(case.spectral_grid.sample_count)
    if count == 1:
        return [start]
    step = (end - start) / float(count - 1)
    return [start + step * index for index in range(count)]


def nearest_grid_values(grid: list[float], targets_nm: tuple[float, ...]) -> list[float]:
    values = []
    for target in targets_nm:
        nearest = min(grid, key=lambda value: abs(value - target))
        if nearest not in values:
            values.append(nearest)
    return values


def instrument_grid_values(grid: list[float]) -> list[float]:
    values = grid[::35] + nearest_grid_values(grid, O2A_MARKERS_NM)
    return sorted(set(values))

## Build The Charts

This cell has no plot output. It keeps the native objects alive while the accessors materialize their chart data.


In [2]:
case = zd.o2a_disamar_reference_input()
oe_baseline.configure_case(case)
grid = spectral_grid(case)
profile_wavelengths_nm = nearest_grid_values(grid, O2A_MARKERS_NM)
response_wavelengths_nm = instrument_grid_values(grid)

with zd.prepare(case) as prepared:
    with prepared.forward_model(jacobian=True) as spectrum:
        noise = components_from_spectrum(
            wavelength_nm=spectrum.wavelength_nm.copy(),
            radiance=spectrum.radiance.copy(),
            irradiance=spectrum.irradiance.copy(),
            reflectance=spectrum.reflectance.copy(),
        )
        noise_table = noise.snr_table()
        reflectance_chart = spectrum.plot.reflectance()
        radiance_chart = spectrum.plot.radiance()
        irradiance_chart = spectrum.plot.irradiance()
        sun_normalized_radiance_chart = spectrum.plot.sun_normalized_radiance()
        aerosol_jacobian_chart = spectrum.plot.jacobian(state="aerosol_optical_depth")
        snr_chart = spectrum.plot.snr(noise_table)
        noise_envelope_chart = spectrum.plot.noise_envelope(noise_table)

    with prepared.atmosphere.budget(wavelengths_nm=grid) as budget:
        optical_depth_chart = budget.plot.optical_depth()

    with prepared.collision_induced_absorption.diagnostics(wavelengths_nm=grid) as cia:
        cia_optical_depth_chart = cia.plot.optical_depth()

    with prepared.instrument_response.sampling_table(
        wavelengths_nm=response_wavelengths_nm
    ) as response:
        isrf_chart = response.plot.curve()

summary = {
    "profile_wavelengths_nm": profile_wavelengths_nm,
    "response_wavelength_count": len(response_wavelengths_nm),
    "noise_table": {
        "snr_wavelengths_nm": noise_table[0],
        "snr_values": noise_table[1],
    },
}
summary

{'profile_wavelengths_nm': [758.0, 760.76, 770.0],
 'response_wavelength_count': 11,
 'noise_table': {'snr_wavelengths_nm': [758.0,
   758.04,
   758.08,
   758.12,
   758.16,
   758.2,
   758.24,
   758.28,
   758.32,
   758.36,
   758.4,
   758.44,
   758.48,
   758.52,
   758.56,
   758.6,
   758.64,
   758.68,
   758.72,
   758.76,
   758.8,
   758.84,
   758.88,
   758.92,
   758.96,
   759.0,
   759.04,
   759.08,
   759.12,
   759.16,
   759.2,
   759.24,
   759.28,
   759.32,
   759.36,
   759.4,
   759.44,
   759.48,
   759.52,
   759.56,
   759.6,
   759.64,
   759.68,
   759.72,
   759.76,
   759.8,
   759.84,
   759.88,
   759.92,
   759.96,
   760.0,
   760.04,
   760.08,
   760.12,
   760.16,
   760.2,
   760.24,
   760.28,
   760.32,
   760.36,
   760.4,
   760.44,
   760.48,
   760.52,
   760.56,
   760.6,
   760.64,
   760.68,
   760.72,
   760.76,
   760.8,
   760.84,
   760.88,
   760.92,
   760.96,
   761.0,
   761.04,
   761.08,
   761.12,
   761.16,
   761.2,
   7

## Reflectance


In [3]:
reflectance_chart

alt.LayerChart(...)

## Radiance


In [4]:
radiance_chart

alt.LayerChart(...)

## Irradiance


In [5]:
irradiance_chart

alt.LayerChart(...)

## Sun-Normalized Radiance


In [6]:
sun_normalized_radiance_chart

alt.LayerChart(...)

## Reflectance Jacobian


In [7]:
aerosol_jacobian_chart

alt.LayerChart(...)

## Signal-To-Noise Ratio


In [8]:
snr_chart

alt.LayerChart(...)

## Noise Envelope


In [9]:
noise_envelope_chart

alt.LayerChart(...)

## Atmospheric Optical Depth


In [10]:
optical_depth_chart

alt.Chart(...)

## O2-O2 CIA Optical Depth


In [11]:
cia_optical_depth_chart

alt.Chart(...)

## ISRF Curve


In [12]:
isrf_chart

alt.Chart(...)